# 02 · QA experiment: does a deterministic calculator reduce numerical errors? (RQ1)
**Modes compared on the same 140 FinanceBench test questions:**
| mode | retrieval | who does arithmetic |
|---|---|---|
| `closed_book` | none | LLM (lower bound) |
| `rag_direct` | top-6 passages | **LLM** (standard RAG baseline) |
| `rag_calc` | top-6 passages | **Python calculator** (LLM writes values + expression) |
| `agent` | tool-calling search / read_page / calculate | **Python calculator** |

**Metrics:** accuracy (numeric match within 1%, LLM judge only for non-numeric gold answers), hallucination rate (wrong + number not found on a cited page or in a calculation), refusal rate, citation correctness, latency p50/p95, cost per request, steps, calculator success rate. **3 repeated runs**; an exact McNemar test compares paired outcomes.

Set your key first (aistudio.google.com). **Model: `gemini-3.5-flash`** (2.5 Flash is closed to new API users). 503 'overloaded' errors are retried with backoff automatically, and a stopped run resumes where it left off (failed questions are retried). Budget: at the 3.5 Flash list price the full run (4 modes × 140 × 3) costs roughly USD 15–30. Groq also works (set `MIZAN_LLM_BASE_URL=https://api.groq.com/openai/v1`).

In [ ]:
# --- Setup (works on Google Colab and locally) ---------------------------------
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not os.path.exists("mizan-ai"):
    subprocess.run(["git", "clone", "https://github.com/usmanbari-ub/mizan-ai.git"], check=True)
ROOT = os.path.abspath("mizan-ai" if IN_COLAB else "../..")
os.chdir(os.path.join(ROOT, "eval"))
sys.path[:0] = [os.path.join(ROOT, "eval"), os.path.join(ROOT, "backend")]
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", os.path.join(ROOT, "backend/requirements.txt")], check=True)
print("working dir:", os.getcwd())

In [ ]:
import os
os.environ.setdefault("MIZAN_LLM_API_KEY", "PASTE_YOUR_KEY")          # or use Colab secrets
os.environ.setdefault("MIZAN_LLM_MODEL", "gemini-3.5-flash")   # use ONE model for every run (backup: gemini-3.5-flash-lite)
os.environ.setdefault("MIZAN_PRICE_IN_PER_M", "1.50"); os.environ.setdefault("MIZAN_PRICE_OUT_PER_M", "9.00")  # 3.5 Flash list price; lite = 0.30 / 2.50
from common import financebench_pdf_dir
try: financebench_pdf_dir()
except FileNotFoundError: subprocess.run([sys.executable, "download_financebench.py"], check=True)

In [ ]:
# Quick smoke test on 5 questions first (costs a few cents at most)
!python run_qa.py --modes rag_direct rag_calc --limit 5 --runs 1 --retriever bm25x

In [ ]:
# Full experiment: 4 modes x 140 questions x 3 runs (resumable: re-run the cell if it stops)
# Results from before 22 Nov (old grounding rule / prompt v1.0) are NOT comparable: delete old results/qa_*.jsonl first
# Free-tier keys have rate limits; it is fine to do --runs 1 now and add runs later.
!python run_qa.py --modes closed_book rag_direct rag_calc agent --runs 3 --retriever bm25x

In [ ]:
import json, pandas as pd
qa = json.load(open("results/summary.json"))["qa"]
tests = qa.pop("paired_tests", {})
cols = ["accuracy_mean","accuracy_std","numeric_accuracy","hallucination_rate","refusal_rate","citation_correct",
        "gold_page_retrieved","latency_p50_s","latency_p95_s","cost_per_request_usd","steps_mean","calc_tool_success_rate","api_failure_rate"]
display(pd.DataFrame(qa).T[cols])
pd.DataFrame(tests).T

In [ ]:
import matplotlib.pyplot as plt
modes = [m for m in ["closed_book","rag_direct","rag_calc","agent"] if m in qa]
acc = [qa[m]["accuracy_mean"] for m in modes]; err = [qa[m]["accuracy_std"] for m in modes]
hal = [qa[m]["hallucination_rate"] for m in modes]
fig, ax = plt.subplots(figsize=(8,4)); x = range(len(modes))
ax.bar([i-.2 for i in x], acc, .4, yerr=err, label="accuracy", color="#2f6f8f")
ax.bar([i+.2 for i in x], hal, .4, label="hallucination rate", color="#c0504d")
ax.set_xticks(list(x)); ax.set_xticklabels(modes); ax.set_ylim(0,1); ax.legend(); ax.set_title("FinanceBench test (n=140), mean ± std over runs")
os.makedirs("results/figures", exist_ok=True); plt.savefig("results/figures/qa_modes.png", dpi=150); plt.show()

## Error analysis: ≥ 10 failures grouped by cause
Automatic first-pass causes (from `run_qa.error_cause`), then **read at least 10 by hand** and fill the "manual note" column for the demo:
`retrieval_miss` · `ungrounded_number (hallucination)` · `wrong_value_or_reasoning_with_evidence` · `calculation_tool_error` · `refusal_after_retrieval_miss` · `refusal_despite_evidence` · `agent_step_limit`

In [ ]:
mode = "rag_calc"
f = pd.DataFrame(json.load(open(f"results/qa_failures_{mode}.json")))
display(f.groupby("cause").size().rename("count").to_frame())
sample = f.groupby("cause").head(4)[["cause","type","question","gold","answer"]]
sample["manual note"] = ""
sample

**Planned fixes** (write the final version after reading the failures):
* retrieval_miss → hybrid + rerank, page ±1 context, agent re-search
* ungrounded numbers → post-check `grounding_check` in the product: answers below 1.0 are shown with a warning and the user is asked to verify
* wrong line item / period → prompt asks for the exact column header; extraction schema stores the period
* refusal despite evidence → few-shot example from the dev split showing a table read